In [ ]:
!pip install great_expectations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 10.7 MB/s eta 0:00:00


In [ ]:
import nbformat

input_path = "/content/your_notebook.ipynb"
output_path = "/content/fixed_notebook.ipynb"

with open(input_path, "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

# Remove broken widget metadata
if "widgets" in nb.metadata:
    del nb.metadata["widgets"]

with open(output_path, "w", encoding="utf-8") as f:
    nbformat.write(nb, f)

print("Saved fixed notebook to:", output_path)

In [ ]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from datetime import datetime, timedelta
import great_expectations as gx
import great_expectations as gx


np.random.seed(42)
print(f"Great Expectations version: {gx.__version__}")
print("All imports successful.")

Great Expectations version: 1.15.2
All imports successful.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Task 1: Generate a Large Synthetic Dataset with Real-world Corruptions

In [ ]:
N_ROWS = 100_000

def generate_clean_dataset(n=N_ROWS):
    rng = np.random.default_rng(42)

    sale_ids = np.arange(1, n + 1)

    # Skewed customer distribution (power-law)
    customer_ids = rng.zipf(a=1.5, size=n) % 10_000 + 1

    # Skewed product distribution
    product_ids = rng.zipf(a=1.8, size=n) % 500 + 1

    # Timestamps spread across last 365 days
    base_date = datetime(2026, 3, 1)
    offsets = rng.integers(0, 365, size=n)
    hours = rng.integers(0, 24, size=n)
    minutes = rng.integers(0, 60, size=n)
    sale_timestamps = [
        base_date - timedelta(days=int(d), hours=int(h), minutes=int(m))
        for d, h, m in zip(offsets, hours, minutes)
    ]

    # Mostly small quantities with rare large orders
    quantity = rng.choice(
        [1, 2, 3, 4, 5, 10, 20, 50, 100],
        size=n,
        p=[0.30, 0.25, 0.15, 0.10, 0.08, 0.05, 0.04, 0.02, 0.01]
    )

    # Amounts loosely correlated with quantity
    base_price = rng.uniform(5.0, 200.0, size=n)
    amount = np.round(quantity * base_price * rng.uniform(0.9, 1.1, size=n), 2)

    # Sales channels
    channels = rng.choice(
        ["Online", "In-Store", "Phone", "Partner"],
        size=n,
        p=[0.45, 0.35, 0.10, 0.10]
    )

    # Currencies
    currencies = rng.choice(
        ["USD", "EUR", "GBP", "PKR"],
        size=n,
        p=[0.50, 0.25, 0.15, 0.10]
    )

    df = pd.DataFrame({
        "sale_id": sale_ids,
        "customer_id": customer_ids.astype(int),
        "product_id": product_ids.astype(int),
        "sale_ts": sale_timestamps,
        "quantity": quantity.astype(int),
        "amount": amount,
        "channel": channels,
        "currency": currencies
    })

    return df

df_clean = generate_clean_dataset()
print(f"Clean dataset shape: {df_clean.shape}")
print(df_clean.head(10))
print(f"\nData types:\n{df_clean.dtypes}")
print(f"\nBasic stats:\n{df_clean.describe()}")

Clean dataset shape: (100000, 8)
   sale_id  customer_id  product_id             sale_ts  quantity   amount  \
0        1           20           5 2025-06-20 17:31:00        10   680.27   
1        2            2           2 2025-12-09 12:44:00         3    92.64   
2        3            2           2 2026-02-08 20:00:00         1    21.52   
3        4            4           3 2025-10-22 23:43:00        10  1440.14   
4        5            6           3 2025-04-03 13:22:00         1    12.99   
5        6           18           2 2025-09-06 12:39:00         4   323.01   
6        7           21           2 2025-10-13 00:05:00         1    57.18   
7        8            4           5 2025-06-23 00:33:00         1    74.10   
8        9            2           4 2026-01-01 21:18:00         4   631.97   
9       10            3           2 2025-06-20 22:54:00         4   308.98   

    channel currency  
0  In-Store      USD  
1  In-Store      USD  
2    Online      USD  
3    Online     

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Generate corrupted dataset ---
def inject_corruptions(df_clean):
    rng = np.random.default_rng(123)
    df = df_clean.copy()
    n = len(df)

    # Inject exact duplicates (~2%)
    n_exact_dup = int(n * 0.02)
    dup_idx = rng.choice(n, size=n_exact_dup, replace=False)
    exact_dups = df.iloc[dup_idx].copy()
    df = pd.concat([df, exact_dups], ignore_index=True)
    print(f"Injected {n_exact_dup} exact duplicates")

    # Inject near-duplicates (~2%)
    n_near_dup = int(n * 0.02)
    near_idx = rng.choice(n, size=n_near_dup, replace=False)
    near_dups = df_clean.iloc[near_idx].copy()
    # Small timestamp shifts
    near_dups["sale_ts"] = near_dups["sale_ts"] + pd.to_timedelta(
        rng.integers(1, 60, size=n_near_dup), unit="s"
    )
    # Slight quantity changes
    qty_mask = rng.random(n_near_dup) < 0.3
    near_dups.loc[qty_mask, "quantity"] = near_dups.loc[qty_mask, "quantity"] + rng.choice([-1, 1], size=qty_mask.sum())
    # Whitespace/casing in channel
    ch_mask = rng.random(n_near_dup) < 0.4
    near_dups.loc[ch_mask, "channel"] = near_dups.loc[ch_mask, "channel"].str.upper()
    df = pd.concat([df, near_dups], ignore_index=True)
    print(f"Injected {n_near_dup} near-duplicates")

    current_n = len(df)

    # Inject missing values (~5%)
    n_missing = int(current_n * 0.05)
    for col in ["customer_id", "product_id", "sale_ts", "quantity", "amount", "channel"]:
        miss_idx = rng.choice(current_n, size=n_missing // 6, replace=False)
        if col in ["customer_id", "product_id", "quantity", "amount"]:
            df.loc[miss_idx, col] = np.nan
        elif col == "channel":
            placeholder = rng.choice(["N/A", "unknown", "", None], size=len(miss_idx))
            for i, idx in enumerate(miss_idx):
                df.at[idx, col] = placeholder[i]
        else:
            df.loc[miss_idx, col] = pd.NaT
    print(f"Injected missing values in ~{n_missing} cells")

    # Inject invalid ranges and outliers (~3%)
    n_invalid = int(current_n * 0.03)
    # Negative quantities
    neg_qty_idx = rng.choice(current_n, size=n_invalid // 3, replace=False)
    df.loc[neg_qty_idx, "quantity"] = rng.integers(-100, 0, size=len(neg_qty_idx)).astype(float)
    # Zero quantities
    zero_qty_idx = rng.choice(current_n, size=n_invalid // 6, replace=False)
    df.loc[zero_qty_idx, "quantity"] = 0
    # Extreme amounts
    extreme_idx = rng.choice(current_n, size=n_invalid // 3, replace=False)
    df.loc[extreme_idx, "amount"] = rng.uniform(100_000, 10_000_000, size=len(extreme_idx))
    # Negative amounts
    neg_amt_idx = rng.choice(current_n, size=n_invalid // 6, replace=False)
    df.loc[neg_amt_idx, "amount"] = rng.uniform(-5000, -1, size=len(neg_amt_idx))
    print(f"Injected invalid ranges and outliers in ~{n_invalid} rows")

    # Inject timestamp anomalies (~2%)
    n_ts = int(current_n * 0.02)
    # Future timestamps
    future_idx = rng.choice(current_n, size=n_ts // 2, replace=False)
    df.loc[future_idx, "sale_ts"] = pd.Timestamp("2028-06-15") + pd.to_timedelta(
        rng.integers(0, 365, size=len(future_idx)), unit="D"
    )
    # Very old timestamps (10-20 years ago)
    old_idx = rng.choice(current_n, size=n_ts // 2, replace=False)
    df.loc[old_idx, "sale_ts"] = pd.Timestamp("2010-01-01") - pd.to_timedelta(
        rng.integers(0, 3650, size=len(old_idx)), unit="D"
    )
    print(f"Injected {n_ts} timestamp anomalies")

    # Inject referential issues
    n_ref = int(current_n * 0.02)
    bad_cust_idx = rng.choice(current_n, size=n_ref // 2, replace=False)
    df.loc[bad_cust_idx, "customer_id"] = rng.integers(90_000, 99_999, size=len(bad_cust_idx)).astype(float)
    bad_prod_idx = rng.choice(current_n, size=n_ref // 2, replace=False)
    df.loc[bad_prod_idx, "product_id"] = rng.integers(9_000, 9_999, size=len(bad_prod_idx)).astype(float)
    print(f"Injected {n_ref} referential integrity issues")

    # Inject currency formatting issues
    n_fmt = int(current_n * 0.04)
    fmt_idx = rng.choice(current_n, size=n_fmt, replace=False)
    df["amount"] = df["amount"].astype(object)
    for i, idx in enumerate(fmt_idx):
        val = df.at[idx, "amount"]
        if pd.notna(val):
            choice = rng.integers(0, 4)
            if choice == 0:
                df.at[idx, "amount"] = f"${val}"
            elif choice == 1:
                df.at[idx, "amount"] = f"€{val}"
            elif choice == 2:
                df.at[idx, "amount"] = str(val).replace(".", ",")
            else:
                df.at[idx, "amount"] = f" {val} USD "
    print(f"Injected {n_fmt} currency formatting issues")

    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    return df

df_corrupted = inject_corruptions(df_clean)
print(f"\nCorrupted dataset shape: {df_corrupted.shape}")
print(df_corrupted.head(10))

Injected 2000 exact duplicates
Injected 2000 near-duplicates
Injected missing values in ~5200 cells
Injected invalid ranges and outliers in ~3120 rows
Injected 2080 timestamp anomalies
Injected 2080 referential integrity issues
Injected 4160 currency formatting issues

Corrupted dataset shape: (104000, 8)
   sale_id  customer_id  product_id             sale_ts  quantity   amount  \
0    65901          2.0         2.0 2025-08-30 09:17:00       2.0   216.33   
1    42135          7.0        28.0 2025-06-11 22:58:00      50.0  9146.32   
2    80135          2.0         2.0 2025-12-05 13:51:00       3.0    82.29   
3    20249          2.0         2.0 2025-10-24 10:41:00       3.0    216.2   
4    65468          5.0         2.0 2025-05-31 11:48:00       0.0   145.29   
5    99231          3.0         3.0 2025-07-18 14:56:00     100.0  5137.56   
6    94846          5.0         2.0 2025-03-03 15:53:00       1.0    27.39   
7    31250          4.0         2.0 2025-08-31 03:23:00       1.0   1

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Save both datasets as CSV
df_clean.to_csv("clean_synthetic_dataset.csv", index=False)
df_corrupted.to_csv("corrupted_synthetic_dataset.csv", index=False)
print("Saved: clean_synthetic_dataset.csv")
print("Saved: corrupted_synthetic_dataset.csv")
print(f"\nClean rows: {len(df_clean)} | Corrupted rows: {len(df_corrupted)}")

Saved: clean_synthetic_dataset.csv
Saved: corrupted_synthetic_dataset.csv

Clean rows: 100000 | Corrupted rows: 104000


## Task 2: Data Quality Report with Great Expectations on Synthetic Data


In [ ]:
# Helper functions
def gx_result_to_dict(result):
    if isinstance(result, dict):
        return result
    for method_name in ("to_json_dict", "as_dict", "model_dump"):
        method = getattr(result, method_name, None)
        if callable(method):
            try:
                converted = method()
                if isinstance(converted, dict):
                    return converted
            except Exception:
                pass
    for method_name in ("to_json", "model_dump_json", "json"):
        method = getattr(result, method_name, None)
        if callable(method):
            try:
                raw = method()
                if isinstance(raw, str):
                    return json.loads(raw)
            except Exception:
                pass
    try:
        return dict(result)
    except Exception:
        return {"repr": repr(result)}

def summarize_result(result_dict):
    config = result_dict.get("expectation_config", {})
    res = result_dict.get("result", {})
    return {
        "expectation_type": config.get("type") or config.get("expectation_type", "unknown"),
        "success": result_dict.get("success"),
        "unexpected_count": res.get("unexpected_count"),
        "unexpected_percent": res.get("unexpected_percent"),
        "partial_unexpected_list": res.get("partial_unexpected_list", []),
    }

def setup_gx_batch(df, source_name, asset_name):
    context = gx.get_context()
    data_source = context.data_sources.add_pandas(name=source_name)
    data_asset = data_source.add_dataframe_asset(name=asset_name)
    batch_def = data_asset.add_batch_definition_whole_dataframe(f"whole_{asset_name}")
    batch = batch_def.get_batch(batch_parameters={"dataframe": df})
    return context, batch_def, batch

print("GX helper functions defined.")

GX helper functions defined.


In [ ]:
# Define expectations for the synthetic sales dataset
def build_synthetic_expectations():
    expectations = [
        # Schema enforcement
        gx.expectations.ExpectColumnToExist(column="sale_id"),
        gx.expectations.ExpectColumnToExist(column="customer_id"),
        gx.expectations.ExpectColumnToExist(column="product_id"),
        gx.expectations.ExpectColumnToExist(column="sale_ts"),
        gx.expectations.ExpectColumnToExist(column="quantity"),
        gx.expectations.ExpectColumnToExist(column="amount"),
        gx.expectations.ExpectColumnToExist(column="channel"),
        gx.expectations.ExpectColumnToExist(column="currency"),

        # Completeness checks
        gx.expectations.ExpectColumnValuesToNotBeNull(column="sale_id"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="customer_id", mostly=0.95),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="product_id", mostly=0.95),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="sale_ts", mostly=0.95),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="quantity", mostly=0.95),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="amount", mostly=0.90),

        # Uniqueness
        gx.expectations.ExpectColumnValuesToBeUnique(column="sale_id"),

        # Value range checks
        gx.expectations.ExpectColumnValuesToBeBetween(column="quantity", min_value=1, max_value=200),
        gx.expectations.ExpectColumnValuesToBeBetween(column="customer_id", min_value=1, max_value=10000),
        gx.expectations.ExpectColumnValuesToBeBetween(column="product_id", min_value=1, max_value=500),

        # Row count check
        gx.expectations.ExpectTableRowCountToBeBetween(min_value=90000, max_value=120000),

        # Controlled vocabulary
        gx.expectations.ExpectColumnValuesToBeInSet(
            column="channel",
            value_set=["Online", "In-Store", "Phone", "Partner"]
        ),
        gx.expectations.ExpectColumnValuesToBeInSet(
            column="currency",
            value_set=["USD", "EUR", "GBP", "PKR"]
        ),

        # Type checks
        gx.expectations.ExpectColumnValuesToBeOfType(column="quantity", type_="float64"),
    ]
    return expectations

synthetic_expectations = build_synthetic_expectations()
print(f"Defined {len(synthetic_expectations)} expectations for synthetic data")

Defined 22 expectations for synthetic data


In [ ]:
# Validate CLEAN synthetic dataset
print("VALIDATING CLEAN SYNTHETIC DATASET")

context_clean, batch_def_clean, batch_clean = setup_gx_batch(
    df_clean, "clean_source", "clean_sales_asset"
)

clean_summaries = []
for exp in synthetic_expectations:
    result = batch_clean.validate(exp)
    result_dict = gx_result_to_dict(result)
    summary = summarize_result(result_dict)
    clean_summaries.append(summary)
    status = "PASS" if summary["success"] else "FAIL"
    print(f"  [{status}] {summary['expectation_type']} — unexpected: {summary['unexpected_count']}")

clean_report = pd.DataFrame(clean_summaries)
print(f"\nClean dataset: {clean_report['success'].sum()}/{len(clean_report)} expectations passed")
clean_report.to_csv("clean_validation_report.csv", index=False)
print("Saved: clean_validation_report.csv")

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpnnq3s50a' for ephemeral docs site


VALIDATING CLEAN SYNTHETIC DATASET


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

  [PASS] expect_column_to_exist — unexpected: None


...

  [PASS] expect_column_values_to_be_in_set — unexpected: 0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

  [FAIL] expect_column_values_to_be_of_type — unexpected: None

Clean dataset: 21/22 expectations passed
Saved: clean_validation_report.csv


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Validate CORRUPTED synthetic dataset
print("VALIDATING CORRUPTED SYNTHETIC DATASET")

# For corrupted data: convert amount back to numeric where possible for range checks
df_corrupted_numeric = df_corrupted.copy()
df_corrupted_numeric["amount"] = pd.to_numeric(df_corrupted_numeric["amount"], errors="coerce")
df_corrupted_numeric["quantity"] = pd.to_numeric(df_corrupted_numeric["quantity"], errors="coerce")
df_corrupted_numeric["customer_id"] = pd.to_numeric(df_corrupted_numeric["customer_id"], errors="coerce")
df_corrupted_numeric["product_id"] = pd.to_numeric(df_corrupted_numeric["product_id"], errors="coerce")

context_dirty, batch_def_dirty, batch_dirty = setup_gx_batch(
    df_corrupted_numeric, "dirty_source", "dirty_sales_asset"
)

dirty_summaries = []
for exp in synthetic_expectations:
    result = batch_dirty.validate(exp)
    result_dict = gx_result_to_dict(result)
    summary = summarize_result(result_dict)
    dirty_summaries.append(summary)
    status = "PASS" if summary["success"] else "FAIL"
    print(f"  [{status}] {summary['expectation_type']} — unexpected: {summary['unexpected_count']}")
    if summary["partial_unexpected_list"]:
        print(f"         Examples: {summary['partial_unexpected_list'][:5]}")

dirty_report = pd.DataFrame(dirty_summaries)
print(f"\nCorrupted dataset: {dirty_report['success'].sum()}/{len(dirty_report)} expectations passed")
dirty_report.to_csv("corrupted_validation_report.csv", index=False)
print("Saved: corrupted_validation_report.csv")

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpqox1wukh' for ephemeral docs site


VALIDATING CORRUPTED SYNTHETIC DATASET


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

  [PASS] expect_column_to_exist — unexpected: None


...

  [PASS] expect_column_values_to_be_in_set — unexpected: 0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  [PASS] expect_column_values_to_be_of_type — unexpected: None

Corrupted dataset: 17/22 expectations passed
Saved: corrupted_validation_report.csv


In [ ]:
# Comparison summary
comparison = pd.DataFrame({
    "Expectation": clean_report["expectation_type"],
    "Clean_Pass": clean_report["success"],
    "Corrupted_Pass": dirty_report["success"],
    "Clean_Unexpected": clean_report["unexpected_count"],
    "Corrupted_Unexpected": dirty_report["unexpected_count"],
})
print("COMPARISON: Clean vs Corrupted Validation Results")
print(comparison.to_string(index=False))

comparison.to_csv("clean_vs_corrupted_comparison.csv", index=False)
print("\nSaved: clean_vs_corrupted_comparison.csv")

COMPARISON: Clean vs Corrupted Validation Results
                         Expectation  Clean_Pass  Corrupted_Pass  Clean_Unexpected  Corrupted_Unexpected
              expect_column_to_exist        True            True               NaN                   NaN
              expect_column_to_exist        True            True               NaN                   NaN
              expect_column_to_exist        True            True               NaN                   NaN
              expect_column_to_exist        True            True               NaN                   NaN
              expect_column_to_exist        True            True               NaN                   NaN
              expect_column_to_exist        True            True               NaN                   NaN
              expect_column_to_exist        True            True               NaN                   NaN
              expect_column_to_exist        True            True               NaN                   NaN
 expe

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Task 3: Data Quality Report with Great Expectations on PakWheels Dataset


In [ ]:
# Load PakWheels dataset
df_pw = pd.read_csv("data-a2 (1).csv")
print(f"PakWheels dataset shape: {df_pw.shape}")
print(f"\nColumns: {list(df_pw.columns)}")
print(f"\nData types:\n{df_pw.dtypes}")
print(f"\nMissing values:\n{df_pw.isnull().sum()}")
print(f"\nMissing percentages:\n{(df_pw.isnull().mean() * 100).round(2)}")
print(f"\nBasic stats:\n{df_pw.describe()}")
print(f"\nSample rows:")
df_pw.head()

PakWheels dataset shape: (62302, 14)

Columns: ['addref', 'city', 'assembly', 'body', 'make', 'model', 'year', 'engine', 'transmission', 'fuel', 'color', 'registered', 'mileage', 'price']

Data types:
addref            int64
city             object
assembly         object
body             object
make             object
model            object
year            float64
engine          float64
transmission     object
fuel             object
color            object
registered       object
mileage           int64
price           float64
dtype: object

Missing values:
addref              0
city                0
assembly        42928
body             7091
make                0
model               0
year             3786
engine              3
transmission        0
fuel              709
color            1192
registered          0
mileage             0
price             472
dtype: int64

Missing percentages:
addref           0.00
city             0.00
assembly        68.90
body            11.38
m

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,addref,city,assembly,body,make,model,year,engine,transmission,fuel,color,registered,mileage,price
0,7642697,Islamabad,NaN,Compact SUV,KIA,Sorento,2021.0,3500.0,Automatic,Petrol,NaN,Islamabad,54654,9300000.0
1,7909608,Sadiqabad,Imported,Hatchback,Daihatsu,Mira,2020.0,660.0,Automatic,Petrol,White,Punjab,10000,3700000.0
2,7929224,Peshawar,Imported,Hatchback,Toyota,Vitz,2018.0,1000.0,Automatic,Petrol,Silver,Islamabad,123000,4150000.0
3,7832366,Lahore,NaN,Sedan,Toyota,Corolla,2019.0,1600.0,Automatic,Petrol,White,Lahore,105000,4850000.0
4,7866725,Islamabad,NaN,Sedan,Toyota,Corolla,2022.0,1600.0,Automatic,Petrol,White,Islamabad,6500,6600000.0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Define expectations for PakWheels dataset based on domain knowledge
def build_pakwheels_expectations():
    expectations = [
        # Schema enforcement
        gx.expectations.ExpectColumnToExist(column="city"),
        gx.expectations.ExpectColumnToExist(column="body"),
        gx.expectations.ExpectColumnToExist(column="make"),
        gx.expectations.ExpectColumnToExist(column="model"),
        gx.expectations.ExpectColumnToExist(column="year"),
        gx.expectations.ExpectColumnToExist(column="engine"),
        gx.expectations.ExpectColumnToExist(column="transmission"),
        gx.expectations.ExpectColumnToExist(column="fuel"),
        gx.expectations.ExpectColumnToExist(column="mileage"),
        gx.expectations.ExpectColumnToExist(column="price"),

        # Completeness checks
        gx.expectations.ExpectColumnValuesToNotBeNull(column="city"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="make"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="model"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="transmission"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="mileage"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="year", mostly=0.90),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="engine", mostly=0.95),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="price", mostly=0.95),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="body", mostly=0.85),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="fuel", mostly=0.95),

        # Value range checks (realistic for used cars in Pakistan)
        gx.expectations.ExpectColumnValuesToBeBetween(column="year", min_value=1970, max_value=2026),
        gx.expectations.ExpectColumnValuesToBeBetween(column="engine", min_value=50, max_value=8000),
        gx.expectations.ExpectColumnValuesToBeBetween(column="mileage", min_value=0, max_value=1_000_000),
        gx.expectations.ExpectColumnValuesToBeBetween(column="price", min_value=50_000, max_value=200_000_000),

        # Controlled vocabulary checks
        gx.expectations.ExpectColumnValuesToBeInSet(
            column="transmission",
            value_set=["Manual", "Automatic"]
        ),
        gx.expectations.ExpectColumnValuesToBeInSet(
            column="fuel",
            value_set=["Petrol", "Diesel", "Hybrid", "CNG", "LPG", "Electric"]
        ),
        gx.expectations.ExpectColumnValuesToBeInSet(
            column="body",
            value_set=[
                "Sedan", "Hatchback", "SUV", "Crossover", "Van", "Truck",
                "Compact SUV", "Compact sedan", "Mini Van", "Mini Vehicles",
                "Double Cabin", "Convertible", "Coupe", "Station Wagon",
                "Off-Road Vehicles", "MPV", "Micro Van", "Pick Up",
                "High Roof", "Compact Hatchback", "Compact Van", "Subcompact Hatchback"
            ],
            mostly=0.90
        ),

        # Row count check
        gx.expectations.ExpectTableRowCountToBeBetween(min_value=50000, max_value=100000),

        # Uniqueness of ad reference
        gx.expectations.ExpectColumnValuesToBeUnique(column="addref"),
    ]
    return expectations

pw_expectations = build_pakwheels_expectations()
print(f"Defined {len(pw_expectations)} expectations for PakWheels dataset")

Defined 29 expectations for PakWheels dataset


In [ ]:
# Validate PakWheels dataset
print("VALIDATING PAKWHEELS DATASET")

context_pw, batch_def_pw, batch_pw = setup_gx_batch(
    df_pw, "pakwheels_source", "pakwheels_asset"
)

pw_summaries = []
for exp in pw_expectations:
    result = batch_pw.validate(exp)
    result_dict = gx_result_to_dict(result)
    summary = summarize_result(result_dict)
    pw_summaries.append(summary)
    status = "PASS" if summary["success"] else "FAIL"
    print(f"  [{status}] {summary['expectation_type']} — unexpected: {summary['unexpected_count']}")
    if not summary["success"] and summary["partial_unexpected_list"]:
        print(f"         Examples: {summary['partial_unexpected_list'][:5]}")

pw_report = pd.DataFrame(pw_summaries)
print(f"\nPakWheels dataset: {pw_report['success'].sum()}/{len(pw_report)} expectations passed")
pw_report.to_csv("pakwheels_validation_report.csv", index=False)
print("Saved: pakwheels_validation_report.csv")

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpaen5phaz' for ephemeral docs site


VALIDATING PAKWHEELS DATASET


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

  [PASS] expect_column_to_exist — unexpected: None


...

  [PASS] expect_table_row_count_to_be_between — unexpected: None


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  [PASS] expect_column_values_to_be_unique — unexpected: 0

PakWheels dataset: 28/29 expectations passed
Saved: pakwheels_validation_report.csv


## Task 4: Data Cleaning and Re-validation


In [ ]:
# --- Clean corrupted synthetic dataset ---
def clean_corrupted_synthetic(df):
    df_c = df.copy()
    print(f"Starting shape: {df_c.shape}")

    # 1. Fix currency formatting in amount column
    def parse_amount(val):
        if pd.isna(val):
            return np.nan
        s = str(val).strip()
        for char in ["$", "€", "£", "USD", "EUR", "GBP", "PKR"]:
            s = s.replace(char, "")
        s = s.strip()
        if "," in s and "." not in s:
            s = s.replace(",", ".")
        try:
            return float(s)
        except (ValueError, TypeError):
            return np.nan
    df_c["amount"] = df_c["amount"].apply(parse_amount)
    print(f"  Fixed currency formatting in amount")

    # 2. Convert columns to numeric
    for col in ["quantity", "customer_id", "product_id"]:
        df_c[col] = pd.to_numeric(df_c[col], errors="coerce")

    # 3. Remove exact duplicates
    before = len(df_c)
    df_c = df_c.drop_duplicates()
    print(f"  Removed {before - len(df_c)} exact duplicate rows")

    # 4. Handle missing values
    df_c["customer_id"] = df_c["customer_id"].fillna(df_c["customer_id"].median())
    df_c["product_id"] = df_c["product_id"].fillna(df_c["product_id"].median())
    df_c["quantity"] = df_c["quantity"].fillna(df_c["quantity"].median())
    df_c["amount"] = df_c["amount"].fillna(df_c["amount"].median())
    df_c["sale_ts"] = df_c["sale_ts"].fillna(method="ffill")
    # Fix channel placeholders
    df_c["channel"] = df_c["channel"].replace({"N/A": None, "unknown": None, "": None})
    df_c["channel"] = df_c["channel"].fillna("Online")
    print(f"  Imputed missing values")

    # 5. Standardize channel values (fix casing)
    channel_map = {
        "ONLINE": "Online", "IN-STORE": "In-Store", "PHONE": "Phone",
        "PARTNER": "Partner", "online": "Online", "in-store": "In-Store"
    }
    df_c["channel"] = df_c["channel"].replace(channel_map)
    print(f"  Standardized channel values")

    # 6. Fix invalid ranges
    df_c.loc[df_c["quantity"] <= 0, "quantity"] = 1
    df_c.loc[df_c["amount"] < 0, "amount"] = df_c.loc[df_c["amount"] >= 0, "amount"].median()
    amount_cap = df_c["amount"].quantile(0.99)
    df_c.loc[df_c["amount"] > amount_cap, "amount"] = amount_cap
    print(f"  Fixed invalid quantity and amount ranges")

    # 7. Fix referential issues (cap IDs to valid ranges)
    df_c.loc[df_c["customer_id"] > 10000, "customer_id"] = df_c.loc[
        df_c["customer_id"] <= 10000, "customer_id"
    ].median()
    df_c.loc[df_c["product_id"] > 500, "product_id"] = df_c.loc[
        df_c["product_id"] <= 500, "product_id"
    ].median()
    print(f"  Fixed referential integrity issues")

    # 8. Fix timestamp anomalies
    now = pd.Timestamp("2026-03-30")
    one_year_ago = now - pd.Timedelta(days=400)
    df_c["sale_ts"] = pd.to_datetime(df_c["sale_ts"], errors="coerce")
    df_c.loc[df_c["sale_ts"] > now, "sale_ts"] = now - pd.Timedelta(days=np.random.randint(1, 365))
    df_c.loc[df_c["sale_ts"] < pd.Timestamp("2020-01-01"), "sale_ts"] = one_year_ago
    print(f"  Fixed timestamp anomalies")

    # Convert IDs back to int
    df_c["customer_id"] = df_c["customer_id"].astype(int)
    df_c["product_id"] = df_c["product_id"].astype(int)
    df_c["quantity"] = df_c["quantity"].astype(int)

    print(f"Final shape: {df_c.shape}")
    return df_c

df_corrupted_cleaned = clean_corrupted_synthetic(df_corrupted)
df_corrupted_cleaned.to_csv("cleaned_corrupted_synthetic_dataset.csv", index=False)
print("\nSaved: cleaned_corrupted_synthetic_dataset.csv")

Starting shape: (104000, 8)
  Fixed currency formatting in amount
  Removed 1550 exact duplicate rows


/tmp/ipykernel_4159/1955131202.py:37: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_c["sale_ts"] = df_c["sale_ts"].fillna(method="ffill")


  Imputed missing values
  Standardized channel values
  Fixed invalid quantity and amount ranges
  Fixed referential integrity issues


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  Fixed timestamp anomalies
Final shape: (102450, 8)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



Saved: cleaned_corrupted_synthetic_dataset.csv


### Cleaning the PakWheels Dataset

In [ ]:
# --- Clean PakWheels dataset ---
def clean_pakwheels(df):
    df_c = df.copy()
    print(f"Starting shape: {df_c.shape}")

    # 1. Drop rows with missing price (target variable)
    before = len(df_c)
    df_c = df_c.dropna(subset=["price"])
    print(f"  Dropped {before - len(df_c)} rows with missing price")

    # 2. Drop assembly column (69% missing, only one meaningful value)
    df_c = df_c.drop(columns=["assembly"], errors="ignore")
    print(f"  Dropped assembly column")

    # 3. Remove duplicate addref entries
    before = len(df_c)
    df_c = df_c.drop_duplicates(subset=["addref"], keep="first")
    print(f"  Removed {before - len(df_c)} duplicate addref rows")

    # 4. Handle missing values in numerical columns
    num_cols = ["year", "engine"]
    for c in num_cols:
        n_miss = df_c[c].isnull().sum()
        if n_miss > 0:
            df_c[c] = df_c[c].fillna(df_c[c].median())
            print(f"  Imputed {n_miss} missing values in {c} with median")

    # 5. Handle missing values in categorical columns
    cat_cols = ["body", "fuel", "color", "registered"]
    for c in cat_cols:
        n_miss = df_c[c].isnull().sum()
        if n_miss > 0:
            df_c[c] = df_c[c].fillna(df_c[c].mode().iloc[0])
            print(f"  Imputed {n_miss} missing values in {c} with mode")

    # 6. Fix invalid engine capacity values
    df_c.loc[df_c["engine"] < 50, "engine"] = df_c["engine"].median()
    df_c.loc[df_c["engine"] > 8000, "engine"] = df_c["engine"].median()
    print(f"  Fixed invalid engine capacity values")

    # 7. Fix invalid year values
    df_c.loc[df_c["year"] < 1970, "year"] = df_c["year"].median()
    df_c.loc[df_c["year"] > 2026, "year"] = df_c["year"].median()
    print(f"  Fixed invalid year values")

    # 8. Cap extreme mileage values
    mileage_cap = df_c["mileage"].quantile(0.99)
    df_c.loc[df_c["mileage"] > mileage_cap, "mileage"] = int(mileage_cap)
    print(f"  Capped extreme mileage values at {mileage_cap}")

    # 9. Cap extreme price values
    price_cap = df_c["price"].quantile(0.99)
    df_c.loc[df_c["price"] > price_cap, "price"] = price_cap
    print(f"  Capped extreme price values at {price_cap:,.0f}")

    print(f"Final shape: {df_c.shape}")
    print(f"Remaining missing values:\n{df_c.isnull().sum()}")
    return df_c

df_pw_cleaned = clean_pakwheels(df_pw)
df_pw_cleaned.to_csv("cleaned_pakwheels_dataset.csv", index=False)
print("\nSaved: cleaned_pakwheels_dataset.csv")

Starting shape: (62302, 14)
  Dropped 472 rows with missing price
  Dropped assembly column
  Removed 0 duplicate addref rows
  Imputed 3659 missing values in year with median
  Imputed 3 missing values in engine with median
  Imputed 7056 missing values in body with mode
  Imputed 702 missing values in fuel with mode
  Imputed 1145 missing values in color with mode
  Fixed invalid engine capacity values
  Fixed invalid year values
  Capped extreme mileage values at 388922.489999999
  Capped extreme price values at 23,000,000
Final shape: (61830, 13)
Remaining missing values:
addref          0
city            0
body            0
make            0
model           0
year            0
engine          0
transmission    0
fuel            0
color           0
registered      0
mileage         0
price           0
dtype: int64


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



Saved: cleaned_pakwheels_dataset.csv


### Re-validation after Cleaning

In [ ]:
# Re-validate cleaned corrupted synthetic dataset
print("RE-VALIDATING CLEANED CORRUPTED SYNTHETIC DATASET")

context_cc, batch_def_cc, batch_cc = setup_gx_batch(
    df_corrupted_cleaned, "cleaned_corrupt_source", "cleaned_corrupt_asset"
)

# Rebuild expectations appropriate for cleaned data (quantity is now int, not float)
revalidation_expectations = [
    gx.expectations.ExpectColumnToExist(column="sale_id"),
    gx.expectations.ExpectColumnToExist(column="customer_id"),
    gx.expectations.ExpectColumnToExist(column="quantity"),
    gx.expectations.ExpectColumnToExist(column="amount"),
    gx.expectations.ExpectColumnToExist(column="channel"),
    gx.expectations.ExpectColumnValuesToNotBeNull(column="customer_id"),
    gx.expectations.ExpectColumnValuesToNotBeNull(column="product_id"),
    gx.expectations.ExpectColumnValuesToNotBeNull(column="quantity"),
    gx.expectations.ExpectColumnValuesToNotBeNull(column="amount"),
    gx.expectations.ExpectColumnValuesToNotBeNull(column="channel"),
    gx.expectations.ExpectColumnValuesToBeBetween(column="quantity", min_value=1, max_value=200),
    gx.expectations.ExpectColumnValuesToBeBetween(column="customer_id", min_value=1, max_value=10000),
    gx.expectations.ExpectColumnValuesToBeBetween(column="product_id", min_value=1, max_value=500),
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="channel",
        value_set=["Online", "In-Store", "Phone", "Partner"]
    ),
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="currency",
        value_set=["USD", "EUR", "GBP", "PKR"]
    ),
]

cc_summaries = []
for exp in revalidation_expectations:
    result = batch_cc.validate(exp)
    result_dict = gx_result_to_dict(result)
    summary = summarize_result(result_dict)
    cc_summaries.append(summary)
    status = "PASS" if summary["success"] else "FAIL"
    print(f"  [{status}] {summary['expectation_type']} — unexpected: {summary['unexpected_count']}")

cc_report = pd.DataFrame(cc_summaries)
print(f"\nCleaned corrupted: {cc_report['success'].sum()}/{len(cc_report)} expectations passed")
cc_report.to_csv("cleaned_corrupted_revalidation_report.csv", index=False)
print("Saved: cleaned_corrupted_revalidation_report.csv")

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpbja8kd4z' for ephemeral docs site


RE-VALIDATING CLEANED CORRUPTED SYNTHETIC DATASET


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


...

  [PASS] expect_column_values_to_be_in_set — unexpected: 0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

  [PASS] expect_column_values_to_be_in_set — unexpected: 0

Cleaned corrupted: 15/15 expectations passed
Saved: cleaned_corrupted_revalidation_report.csv


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Re-validate cleaned PakWheels dataset
print("RE-VALIDATING CLEANED PAKWHEELS DATASET")

context_pwc, batch_def_pwc, batch_pwc = setup_gx_batch(
    df_pw_cleaned, "pw_cleaned_source", "pw_cleaned_asset"
)

# Rebuild PakWheels expectations (without assembly/addref changes)
pw_revalidation_expectations = [
    gx.expectations.ExpectColumnToExist(column="city"),
    gx.expectations.ExpectColumnToExist(column="body"),
    gx.expectations.ExpectColumnToExist(column="make"),
    gx.expectations.ExpectColumnToExist(column="year"),
    gx.expectations.ExpectColumnToExist(column="engine"),
    gx.expectations.ExpectColumnToExist(column="transmission"),
    gx.expectations.ExpectColumnToExist(column="fuel"),
    gx.expectations.ExpectColumnToExist(column="price"),
    gx.expectations.ExpectColumnValuesToNotBeNull(column="city"),
    gx.expectations.ExpectColumnValuesToNotBeNull(column="make"),
    gx.expectations.ExpectColumnValuesToNotBeNull(column="model"),
    gx.expectations.ExpectColumnValuesToNotBeNull(column="transmission"),
    gx.expectations.ExpectColumnValuesToNotBeNull(column="year"),
    gx.expectations.ExpectColumnValuesToNotBeNull(column="engine"),
    gx.expectations.ExpectColumnValuesToNotBeNull(column="price"),
    gx.expectations.ExpectColumnValuesToNotBeNull(column="body"),
    gx.expectations.ExpectColumnValuesToNotBeNull(column="fuel"),
    gx.expectations.ExpectColumnValuesToBeBetween(column="year", min_value=1970, max_value=2026),
    gx.expectations.ExpectColumnValuesToBeBetween(column="engine", min_value=50, max_value=8000),
    gx.expectations.ExpectColumnValuesToBeBetween(column="mileage", min_value=0, max_value=1_000_000),
    gx.expectations.ExpectColumnValuesToBeBetween(column="price", min_value=50_000, max_value=200_000_000),
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="transmission",
        value_set=["Manual", "Automatic"]
    ),
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="fuel",
        value_set=["Petrol", "Diesel", "Hybrid", "CNG", "LPG", "Electric"]
    ),
]

pwc_summaries = []
for exp in pw_revalidation_expectations:
    result = batch_pwc.validate(exp)
    result_dict = gx_result_to_dict(result)
    summary = summarize_result(result_dict)
    pwc_summaries.append(summary)
    status = "PASS" if summary["success"] else "FAIL"
    print(f"  [{status}] {summary['expectation_type']} — unexpected: {summary['unexpected_count']}")

pwc_report = pd.DataFrame(pwc_summaries)
print(f"\nCleaned PakWheels: {pwc_report['success'].sum()}/{len(pwc_report)} expectations passed")
pwc_report.to_csv("cleaned_pakwheels_revalidation_report.csv", index=False)
print("Saved: cleaned_pakwheels_revalidation_report.csv")

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmp30ljtmax' for ephemeral docs site


RE-VALIDATING CLEANED PAKWHEELS DATASET


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


...

  [PASS] expect_column_values_to_be_in_set — unexpected: 0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  [PASS] expect_column_values_to_be_in_set — unexpected: 0

Cleaned PakWheels: 23/23 expectations passed
Saved: cleaned_pakwheels_revalidation_report.csv


In [ ]:
# Final comparison: Before vs After cleaning
print("IMPROVEMENT COMPARISON: Before vs After Cleaning")

print("\n--- Corrupted Synthetic Dataset ---")
print(f"Before cleaning: {dirty_report['success'].sum()}/{len(dirty_report)} expectations passed")
print(f"After cleaning:  {cc_report['success'].sum()}/{len(cc_report)} expectations passed")

print("\n--- PakWheels Dataset ---")
print(f"Before cleaning: {pw_report['success'].sum()}/{len(pw_report)} expectations passed")
print(f"After cleaning:  {pwc_report['success'].sum()}/{len(pwc_report)} expectations passed")

IMPROVEMENT COMPARISON: Before vs After Cleaning

--- Corrupted Synthetic Dataset ---
Before cleaning: 17/22 expectations passed
After cleaning:  15/15 expectations passed

--- PakWheels Dataset ---
Before cleaning: 28/29 expectations passed
After cleaning:  23/23 expectations passed
